# RightTime 전처리와 기준선 모델

iPhone RightTime 앱의 `내보내기 파일 만들기`로 받은 **JSON 파일**을 올려서 돌린다.
수집 데이터는 이 노트북에만 올리고 GitHub에는 올리지 않는다.

In [ ]:
!git clone -q https://github.com/kp0om/RightTime.git
%cd RightTime/ml

## 1. 데이터 올리기
`righttime-YYYYMMDD-HHMM.json` 파일을 고른다. (CSV가 아니라 JSON)

In [ ]:
from google.colab import files
uploaded = files.upload()
json_path = next(name for name in uploaded if name.endswith('.json'))
json_path

## 2. 데이터 확인
결과 분포, context가 붙은 비율, T별 양성 비율을 본다.

In [ ]:
from features import build_table, load_prompts, time_split, FEATURE_COLUMNS

prompts, wakes = load_prompts(json_path)
df = time_split(build_table(prompts))
print(len(df), '개,', df.date.min(), '~', df.date.max(), '/ 앱 깨어남', len(wakes), '회')
print(df.outcome.value_counts())
print('가속도 있음:', (df.accel_count > 0).mean().round(2), '/ 활동 기록 있음:', (df.act_now_age_log.notna()).mean().round(2))
df[['delivered_at', 'outcome', 'response_s', 'steps_5m', 'accel_std', 'act_now_stationary', 'label_60s', 'split']].tail(10)

## 3. 기준선 비교 (T = 30, 60, 120초)
다수 클래스, 규칙 기반, Logistic Regression, Random Forest. 날짜 단위 시간순 분할(60/20/20)의 test 결과.

In [ ]:
for t in [30, 60, 120]:
    !python baselines.py {json_path} --label-seconds {t}

## 4. 입력 그룹 ablation
시간·걸음·가속도·활동 그룹을 하나씩 빼고 Random Forest 성능 변화를 본다.

In [ ]:
!python baselines.py {json_path} --label-seconds 60 --ablation